In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import warnings
import proplot as pplt
import numpy as np
from rolling_plot_utils import (
    plot_years_and_seasons,
    get_season,
    get_cmap_plus_white,
    BASELINE_SURVEY_START_MJD,
    MJD_2024,
    YEAR,
    get_sun_ra_at_mjd,
    plot_sun_ra,
    plot_rizexptime_fancy,
    mad,
)
import fitsio

## Look at Raw Sims

In [23]:
from rubin_sim.scheduler.utils import SkyAreaGenerator
nside = 128
pixel_area = (27.483891 / 60)**2

def _comp_area(ebvlim):
    maps_arr = fitsio.read(f"baseline_v4.0_10yrs_ebvlim{ebvlim}_nside128_bins40.fits")

    mind = 39

    sm = SkyAreaGenerator(nside=nside)
    footprints_hp_array, labels = sm.return_maps()
    wfd_msk = (labels == 'lowdust') | (labels == 'virgo')
    hmap = maps_arr[mind, :].copy()
    msk = (~wfd_msk) | (hmap < 15)
    hmap[msk] = np.nan

    return np.sum(np.isfinite(hmap)) * pixel_area

baseline_area = _comp_area("200")

print("| ---------- | -------- | --------- |")
print("| E[B-V] max | area.    | frac lost |")
print("| ---------- | -------- | --------- |")
for ebvlim in ["100", "125", "150", "175", "200"]:
    area = _comp_area(ebvlim)

    print("|%11.3f | %0.2f | %9.3f |" % (int(ebvlim)/1000, area, 1.0 - area / baseline_area))


| ---------- | -------- | --------- |
| E[B-V] max | area.    | frac lost |
| ---------- | -------- | --------- |
|      0.100 | 14172.94 |     0.170 |
|      0.125 | 15365.58 |     0.100 |
|      0.150 | 16159.34 |     0.053 |
|      0.175 | 16712.64 |     0.021 |
|      0.200 | 17071.44 |     0.000 |
